# Chapter 17 &mdash; From Decision Tree to BDD

**Concept 5 of the Chapter 17 decomposition:** *From Decision Tree to BDD: What the Construction Actually Does*

Conceptually build the full exponential tree and share common subexpressions &mdash; but never actually build it.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-From-Decision-Tree-To-BDD/Concept-From-Decision-Tree-To-BDD.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The textbook picture: start from the **full binary decision tree** of depth $N$ &mdash;
$2^{N+1}-1$ nodes &mdash; then apply two reductions until nothing changes:

* **delete** a node whose two children are the same node;
* **merge** two nodes with the same variable and the same children.

That is a clean *definition*, and a hopeless *algorithm*: the starting point is
exponential.

Real BDD packages never build the tree. They construct bottom-up through a
**`mk(var, low, high)`** function that applies both reductions **at creation time**:
return `low` if the children agree, and return the existing node if the triple is
already in the unique table. So the reduced form is the only form that ever exists.

This is the same move as Chapter 10's **smart constructors** &mdash; normalise on
construction rather than afterwards.

## 2. Definitions

### The BDD package

In [ ]:
# --- a minimal BDD package ----------------------------------------------
# A node is either the terminal 0/1, or ('n', var_index, low, high) where
# low is the 0-branch and high the 1-branch.  Hash consing (the `unique`
# table) is what makes the representation canonical: structurally equal
# subgraphs become the SAME Python object, so equality is pointer equality.
ZERO, ONE = 0, 1

class BDD:
    def __init__(self, nvars):
        self.nvars = nvars
        self.unique = {}          # (var, low, high) -> node  -- hash consing
        self.apply_cache = {}

    def mk(self, var, low, high):
        if low is high: return low            # REDUCTION 1: skip a useless test
        key = (var, id(low), id(high), self._k(low), self._k(high))
        if key in self.unique: return self.unique[key]   # REDUCTION 2: share
        node = ('n', var, low, high)
        self.unique[key] = node
        return node

    def _k(self, n):
        return n if n in (ZERO, ONE) else ('n', n[1], self._k(n[2]), self._k(n[3]))

    def var(self, i):
        return self.mk(i, ZERO, ONE)

    def apply(self, op, a, b):
        key = (op, self._k(a), self._k(b))
        if key in self.apply_cache: return self.apply_cache[key]
        if a in (ZERO, ONE) and b in (ZERO, ONE):
            r = ONE if op(bool(a), bool(b)) else ZERO
        else:
            va = a[1] if a not in (ZERO, ONE) else self.nvars
            vb = b[1] if b not in (ZERO, ONE) else self.nvars
            v = min(va, vb)
            al, ah = (a[2], a[3]) if va == v else (a, a)
            bl, bh = (b[2], b[3]) if vb == v else (b, b)
            r = self.mk(v, self.apply(op, al, bl), self.apply(op, ah, bh))
        self.apply_cache[key] = r
        return r

    def NOT(self, a):  return self.apply(lambda x, y: not x, a, a)
    def AND(self, a, b): return self.apply(lambda x, y: x and y, a, b)
    def OR(self, a, b):  return self.apply(lambda x, y: x or y, a, b)
    def XOR(self, a, b): return self.apply(lambda x, y: x != y, a, b)

    def evaluate(self, node, assign):
        while node not in (ZERO, ONE):
            node = node[3] if assign[node[1]] else node[2]
        return bool(node)

    def size(self, node):
        seen = set()
        def walk(n):
            if n in (ZERO, ONE): return
            k = self._k(n)
            if k in seen: return
            seen.add(k); walk(n[2]); walk(n[3])
        walk(node)
        return len(seen)

    def onset(self, node, order=None):
        from itertools import product
        out = []
        for bits in product([False, True], repeat=self.nvars):
            a = {i: bits[i] for i in range(self.nvars)}
            if self.evaluate(node, a):
                out.append(''.join('1' if bits[i] else '0' for i in range(self.nvars)))
        return sorted(out)


def full_tree_size(N): return 2 ** (N + 1) - 1

### An explicit (wasteful) tree builder, for comparison

In [ ]:
def build_tree(f, N, prefix=()):
    if len(prefix) == N:
        return ONE if f(prefix) else ZERO
    return ('t', len(prefix), build_tree(f, N, prefix + (0,)),
                              build_tree(f, N, prefix + (1,)))

def tree_nodes(t):
    if t in (ZERO, ONE): return 1
    return 1 + tree_nodes(t[2]) + tree_nodes(t[3])

<!-- nav-strip -->

---

&larr;&nbsp;[Ch17&nbsp;4.&nbsp;Linearly Sized BDDs, and How Exponentiality Is Hidden](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-Linearly-Sized-BDDs/Concept-Linearly-Sized-BDDs.ipynb) &nbsp;&middot;&nbsp; [**Chapter 17** index](https://github.com/ganeshutah/Jove/blob/master/Chapter17/README.md) &nbsp;&middot;&nbsp; [Ch17&nbsp;6.&nbsp;Canonicity via Myhill–Nerode, Hash Consing, and the Apply Operation](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-Canonicity-And-Apply/Concept-Canonicity-And-Apply.ipynb)&nbsp;&rarr;

---

## 3. Tests

The full tree is exponential, as advertised.

In [ ]:
f = lambda bits: int(sum(bits) % 2 == 0)          # even parity
for N in [2, 3, 4, 8]:
    print("  N=%d : full tree would have %s nodes" % (N, format(full_tree_size(N), ',')))
t = build_tree(f, 4)
print("\nactually built for N=4 :", tree_nodes(t), "nodes")
assert tree_nodes(t) == full_tree_size(4)

The BDD for the same function is linear.

In [ ]:
for N in [2, 4, 8, 16]:
    b = BDD(N)
    xs = [b.var(i) for i in range(N)]
    g = b.NOT(xs[0])
    for x in xs[1:]: g = b.XOR(g, x)
    print("  N=%2d : full tree %-10s BDD %d nodes"
          % (N, format(full_tree_size(N), ','), b.size(g)))
b = BDD(16); xs = [b.var(i) for i in range(16)]
g = b.NOT(xs[0])
for x in xs[1:]: g = b.XOR(g, x)
assert b.size(g) <= 2 * 16

**`mk` applies both reductions at creation time.**

In [ ]:
b = BDD(3)
n1 = b.mk(2, ZERO, ONE)
n2 = b.mk(2, ZERO, ONE)
print("two identical mk calls returned the same node? ", n1 is n2)
assert n1 is n2
n3 = b.mk(1, n1, n1)
print("mk(1, n, n) returned n itself? ", n3 is n1)
assert n3 is n1
print("\nReduction 2 (merge) is the unique table; reduction 1 (delete) is")
print("the `if low is high: return low` line.  Neither needs a pass.")

So the tree is never materialised.

In [ ]:
N = 12
b = BDD(N)
xs = [b.var(i) for i in range(N)]
g = xs[0]
for x in xs[1:]: g = b.OR(g, x)
print("N=%d : full tree would be %s nodes" % (N, format(full_tree_size(N), ',')))
print("      unique table holds %d nodes, BDD has %d" % (len(b.unique), b.size(g)))
assert len(b.unique) < 100

The same idea as Chapter 10's smart constructors.

In [ ]:
print("Chapter 10 : mkAlt/mkCat/mkStar normalise a regular expression as it")
print("             is built, so re-derived expressions are IDENTICAL and")
print("             the derivative automaton terminates.")
print()
print("Chapter 17 : mk() normalises a BDD node as it is built, so equal")
print("             functions are the SAME pointer and equality is O(1).")
print()
print("Normalise on construction.  It is the same trick twice.")

## 4. Exercises


1. Apply the two reductions by hand to the depth-3 tree for majority.
2. Why is the order of the two reductions irrelevant?
3. What would `mk` have to do if you allowed complement edges?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 246 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter17/Concept-From-Decision-Tree-To-BDD')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')